In [27]:
import numpy as np
import pandas as pd
import gurobipy as grb
from gurobipy import GRB

import matplotlib.pyplot as plt

In [28]:
# 创建一个空的模型
model = grb.Model("Integrated_Energy_System")
model.setParam("OutputFlag", 0)
# 定义模型中的参数
# 时间步数，假设24小时
T = 24

# 设备功率和效率等参数（单位：MW 或者单位功率）
# 电解槽（EL）参数
PEL_max = 5  # 电解槽最大功率（MW）
ηEL = 0.75  # 电解槽效率
meh = 0.03954  # 氢气的热值（MWh/kg）
ΔPEL_up = 0.2  # 电解槽爬坡上限比例（MW）
ΔPEL_down = -0.2  # 电解槽爬坡下限（MW）
αEL = 25  # 电解槽运营和维护成本（¥/MWh）

# 氢燃料电池（HFC）参数
HHFC_max = 25  # 氢燃料电池最大功率（kg/h）
ηHFC = 0.95  # 氢燃料电池效率
ηHFC_e = 0.8  # 氢燃料电池电效率
ηHFC_h = 0.15  # 氢燃料电池热效率
ΔHHFC_up = 0.2  # 氢燃料电池功率爬坡上限比例（kg/h）
ΔHHFC_down = -0.2  # 氢燃料电池功率爬坡下限（kg/h）
αHFC = 18  # 氢燃料电池运营和维护成本（¥/MWh）

# 余热锅炉（WHB）参数
PWHB_max = 600  # 余热锅炉最大功率（MW）
ηWHB = 0.75  # 余热锅炉效率
αWHB = 65  # 余热锅炉运营和维护成本（¥/MWh）

αWT = 70  # 风力发电机运营和维护成本（¥/MWh）
αPV = 32  # 光伏发电运营和维护成本（¥/MWh）

# 燃气轮机（GT）参数
PGT_max = 580  # 燃气轮机最大功率（MW）
ηGT = 0.35  # 燃气轮机效率
ηloss = 0.1  # 能量损失率
ΔPGT_up = 0.2  # 燃气轮机爬坡上限比例（MW）
ΔPGT_down = -0.2  # 燃气轮机爬坡下限（MW）
αGT = 12.5  # 燃气轮机运营和维护成本（¥/MWh）

# 燃气锅炉（GB）参数
PGB_max = 1000  # 燃气锅炉最大功率（MW）
ηGB = 0.75  # 燃气锅炉效率
ΔPGB_up = 0.2  # 燃气锅炉爬坡上限比例（MW）
ΔPGB_down = -0.2  # 燃气锅炉爬坡下限（MW）
αGB = 85  # 燃气锅炉运营和维护成本（¥/MWh）

# 甲烷发生器（MR）参数
HMR_max = 1000      # 甲烷发生器最大耗氢量（kg/h）
ηMR = 0.6           # 甲烷发生器效率
ΔHMR_up = 0.2       # 甲烷发生器爬坡上限比例（kg/h）
ΔHMR_down = -0.2    # 甲烷发生器爬坡下限比例（kg/h）
αMR = 19            # 甲烷发生器运营维护成本（¥/MWh）

# MR 碳利用系数（捕集的 CO2 用于甲烷合成）
θ = 0.00135         # CO2 利用系数
lambda_carbon = 250   # 碳交易基准价格（¥/t）
l_carbon = 100        # 阶梯区间长度（t）
alpha_carbon = 0.25   # 阶梯价格增长率

# 碳捕集系统（CCS）参数
ηCCS = 0.65      # CCS捕集效率
eccs = 0.269     # 单位CO2捕集耗电量（MWh/t）
kcs = 1.4        # 单位CO2封存成本（¥/t）
αCCS = 55        # CCS运行维护成本（¥/MWh）

# 初始碳配额系数（kg/kWh）
kappa_buy = 0.68      # 购电碳配额系数
kappa_GT = 0.612      # GT 碳配额系数
kappa_GB = 0.306      # GB 碳配额系数

# 实际碳排放系数（kg/kWh）
kappa_buy_a = 1.08    # 购电实际碳排放系数
kappa_GT_e = 0.195    # GT 发电实际碳排放系数
kappa_GT_h = 0.39     # GT 供热实际碳排放系数
kappa_GB_a = 0.195    # GB 实际碳排放系数


heat_price = 1000     # 外购热价（¥/MWh）

# 电储能（ES1）参数
Pmax_cha1 = 60  # 电储能充电最大功率（MW）
Pmax_dis1 = 60  # 电储能放电最大功率（MW）
Smin1 = 40  # 电储能最小容量（MWh）
Smax1 = 180  # 电储能最大容量（MWh）
ηcha_ES1 = 0.95  # 电储能充电效率
ηdis_ES1 = 0.95  # 电储能放电效率
α1 = 18  # 电储能的运行维护成本（¥/MWh）

# 热储能（ES2）参数
Pmax_cha2 = 60  # 热储能充电最大功率（MW）
Pmax_dis2 = 60  # 热储能放电最大功率（MW）
Smin2 = 40  # 热储能最小容量（MW）
Smax2 = 180  # 热储能最大容量（MW）
ηcha_ES2 = 0.95  # 热储能充电效率
ηdis_ES2 = 0.95  # 热储能放电效率
α2 = 16  # 热储能的运营维护成本（¥/MWh）

# 气储能（ES3）参数
Pmax_cha3 = 30  # 气储能充电最大功率（MW）
Pmax_dis3 = 30  # 气储能放电最大功率（MW）
Smin3 = 20  # 气储能最小容量（MW）
Smax3 = 90  # 气储能最大容量（MW）
ηcha_ES3 = 0.95  # 气储能充电效率
ηdis_ES3 = 0.95  # 气储能放电效率
α3 = 16  # 气储能的运营维护成本（¥/MWh）

# 氢储能（ES4）参数
Pmax_cha4 = 300  # 氢储能充电最大功率（kg/h）
Pmax_dis4 = 300  # 氢储能放电最大功率（kg/h）
Smin4 = 200  # 氢储能最小容量（kg）
Smax4 = 900  # 氢储能最大容量（kg）
ηcha_ES4 = 0.95  # 氢储能充电效率
ηdis_ES4 = 0.95  # 氢储能放电效率
α4 = 16  # 氢储能的运行维护成本（¥/kg）

# ========= 电价（¥/MWh），24 小时序列 =========
# 原始电价（¥/kWh）× 1000 → ¥/MWh
electricity_price = [
    343.8, 343.8, 343.8, 343.8, 343.8, 343.8, 343.8,   # 0:00–7:00
    680.0, 680.0, 680.0, 680.0,                        # 7:00–11:00
    811.0, 811.0, 811.0,                               # 11:00–14:00
    680.0, 680.0, 680.0,                               # 14:00–17:00
    811.0, 811.0, 811.0, 811.0, 811.0,                 # 17:00–22:00
    380.0, 380.0                                      # 22:00–24:00
]

# 氢气价格（¥/MWh）
hydrogen_price = 1500  # 氢气单价（¥/MWh）

# 天然气价格（¥/MWh）
natural_gas_price = 350  # 天然气单价（¥/MWh）
heat_price = 1000     # 外购热价（¥/MWh）
# 风电和光伏的弃电惩罚成本（¥/MWh）
wind_penalty = 200  # 风电弃电惩罚（¥/MWh）
solar_penalty = 200  # 光伏弃电惩罚（¥/MWh）

# 系统负荷（假设为占位）
electric_load = [15, 25, 26, 28, 28, 32, 35, 41, 43, 49, 55, 68, 66, 56, 43, 42, 47, 51, 53, 54, 50, 39, 30, 27]
thermal_load = [45, 24, 38, 48, 51, 58, 43, 58, 68, 83, 80, 84, 70, 68, 65, 62, 65, 88, 50, 48, 45, 55, 63, 50]
hydrogen_load = [80, 66, 60, 50, 50, 55, 55, 100, 152, 136, 80, 70, 70, 70, 90, 90, 80, 70, 70, 70, 70, 70, 70, 70]
gas_load =  [10, 13, 12, 11, 14, 15, 14, 15, 13, 10, 12, 15, 15, 15, 14, 13, 11, 11, 12, 12, 13, 10, 9, 8]

In [29]:
# =========================
# 1) 可再生出力上限 + 鲁棒场景
# =========================
WT_avail = [20, 33, 36, 46, 47, 43, 40, 50, 38, 34, 48, 31, 38, 29, 11, 8, 32, 47, 33, 42, 52, 50, 59, 42.5]
PV_avail = [0, 0, 0, 0, 0, 0, 0, 0, 0, 17, 30, 40, 45, 46, 43, 33, 21, 8, 0, 0, 0, 0, 0, 0]

def build_uncertain_profile(base, rho, bad_hours):
    prof = base.copy()
    for t in bad_hours:
        prof[t] = base[t] * (1 - rho)
    return prof

def top_k_hours(profile, k):
    return sorted(range(len(profile)), key=lambda t: profile[t], reverse=True)[:k]

Gamma_wt = 3
Gamma_pv = 3
rho_w = 0.2
rho_p = 0.2

bad_wt = top_k_hours(WT_avail, Gamma_wt)
bad_pv = top_k_hours(PV_avail, Gamma_pv)

scenarios = {
    "base": {
        "WT": WT_avail[:],
        "PV": PV_avail[:]
    },
    "wt_robust": {
        "WT": build_uncertain_profile(WT_avail, rho_w, bad_wt),
        "PV": PV_avail[:]
    },
    "pv_robust": {
        "WT": WT_avail[:],
        "PV": build_uncertain_profile(PV_avail, rho_p, bad_pv)
    },
    "both_robust": {
        "WT": build_uncertain_profile(WT_avail, rho_w, bad_wt),
        "PV": build_uncertain_profile(PV_avail, rho_p, bad_pv)
    }
}

S = list(scenarios.keys())

# =========================
# 二阶段场景相关变量（补救变量）
# =========================
Pbuy_e = model.addVars(T, S, lb=0.0, name="Pbuy_e")   # 二阶段外购电
Pbuy_g = model.addVars(T, S, lb=0.0, name="Pbuy_g")   # 二阶段外购气
Pbuy_H = model.addVars(T, S, lb=0.0, name="Pbuy_H")   # 二阶段外购氢
Pbuy_h = model.addVars(T, S, lb=0.0, name="Pbuy_h")   # 二阶段外购热

PWT_use = model.addVars(T, S, lb=0.0, name="PWT_use")   # 二阶段风电利用
PWT_cut = model.addVars(T, S, lb=0.0, name="PWT_cut")   # 二阶段风电弃电
PPV_use = model.addVars(T, S, lb=0.0, name="PPV_use")   # 二阶段光伏利用
PPV_cut = model.addVars(T, S, lb=0.0, name="PPV_cut")   # 二阶段光伏弃电

In [30]:
# =========================
# 0) 时间步长（默认 1 小时）
# =========================
dt = 1.0  # hour


# =========================
# 2) 统一“比例爬坡”参数
# =========================
ramp_GT  = abs(ΔPGT_up)
ramp_GB  = abs(ΔPGB_up)
ramp_EL  = abs(ΔPEL_up)
ramp_HFC = abs(ΔHHFC_up)
ramp_MR = abs(ΔHMR_up)
# =========================
# 3) 变量定义
# ========================
# ---- 可再生与弃电 ----

# ---- GT：电功率/气输入/热输出 ----
PGT_e = model.addVars(T, lb=0.0, ub=PGT_max, name="PGT_e")  # GT发电（MW）
PGT_g = model.addVars(T, lb=0.0, name="PGT_g")              # GT耗气（MW）
PGT_h = model.addVars(T, lb=0.0, name="PGT_h")              # GT产热（MW）

# ---- GB：热功率/气输入 ----
PGB_h = model.addVars(T, lb=0.0, ub=PGB_max, name="PGB_h")  # GB产热（MW）
PGB_g = model.addVars(T, lb=0.0, name="PGB_g")              # GB耗气（MW）

# ---- WHB：回收热 ----
PWHB_h = model.addVars(T, lb=0.0, ub=PWHB_max, name="PWHB_h")  # WHB回收热（MW）

# ---- EL：耗电与产氢 ----
PEL_e = model.addVars(T, lb=0.0, ub=PEL_max, name="PEL_e")  # EL耗电（MW）
HEL   = model.addVars(T, lb=0.0, name="HEL")                # EL产氢（kg/h）

# ---- HFC：耗氢与电/热输出 ----
HHFC    = model.addVars(T, lb=0.0, ub=HHFC_max, name="HHFC")      # HFC耗氢（kg/h）
PHFC_tot = model.addVars(T, lb=0.0, name="PHFC_tot")              # HFC总能量输出（MW）
PHFC_e   = model.addVars(T, lb=0.0, name="PHFC_e")                # HFC电输出（MW）
PHFC_h   = model.addVars(T, lb=0.0, name="PHFC_h")                # HFC热输出（MW）

# ---- MR：耗氢与产甲烷（天然气）----
HMR   = model.addVars(T, lb=0.0, ub=HMR_max, name="HMR")        # MR耗氢（kg/h）
PMR_g = model.addVars(T, lb=0.0, name="PMR_g")                  # MR产天然气（MW）
EMR   = model.addVars(T, lb=0.0, name="EMR")                    # MR消耗CO2（t 或按你后续统一口径）

# ---- CCS：捕碳、未捕集碳、耗电、封存 ----
ECO2      = model.addVars(T, lb=0.0, name="ECO2")               # GT+GB产生CO2
Ecap_CCS  = model.addVars(T, lb=0.0, name="Ecap_CCS")           # CCS捕集CO2
Ecs_CCS   = model.addVars(T, lb=0.0, name="Ecs_CCS")            # 未被捕集的CO2
PCSSt     = model.addVars(T, lb=0.0, name="PCCS")               # CCS耗电（MW）
ECS       = model.addVars(T, lb=0.0, name="ECS")                # 最终封存CO2

# ---- 碳配额 / 实际排放 / 碳交易 ----
Egrid_q = model.addVars(T, lb=0.0, name="Egrid_q")              # 购电初始碳配额
EGT_q   = model.addVars(T, lb=0.0, name="EGT_q")                # GT初始碳配额
EGB_q   = model.addVars(T, lb=0.0, name="EGB_q")                # GB初始碳配额

Egrid_a = model.addVars(T, lb=0.0, name="Egrid_a")              # 购电实际排放
EGT_a   = model.addVars(T, lb=0.0, name="EGT_a")                # GT实际排放
EGB_a   = model.addVars(T, lb=0.0, name="EGB_a")                # GB实际排放
ECO2_env = model.addVars(T, lb=0.0, name="ECO2_env")            # 排入环境CO2

E_IES_quota = model.addVar(lb=0.0, name="E_IES_quota")          # 总初始碳配额
E_IES_actual = model.addVar(lb=-GRB.INFINITY, name="E_IES_actual")  # 参与碳交易的净排放量
CCT = model.addVar(lb=-GRB.INFINITY, name="CarbonTradingCost")  # 碳交易成本

# ---- 储能：ES1 电（MWh/MW）----
Pcha1 = model.addVars(T, lb=0.0, ub=Pmax_cha1, name="Pcha_ES1")   # 充电功率（MW）
Pdis1 = model.addVars(T, lb=0.0, ub=Pmax_dis1, name="Pdis_ES1")   # 放电功率（MW）
bcha1 = model.addVars(T, vtype=GRB.BINARY, name="bcha_ES1")       # 充电状态
bdis1 = model.addVars(T, vtype=GRB.BINARY, name="bdis_ES1")       # 放电状态

# ---- 储能：ES2 热（MWh/MW）----
Pcha2 = model.addVars(T, lb=0.0, ub=Pmax_cha2, name="Pcha_ES2")
Pdis2 = model.addVars(T, lb=0.0, ub=Pmax_dis2, name="Pdis_ES2")
bcha2 = model.addVars(T, vtype=GRB.BINARY, name="bcha_ES2")
bdis2 = model.addVars(T, vtype=GRB.BINARY, name="bdis_ES2")

# ---- 储能：ES3 气（MWh/MW）----
Pcha3 = model.addVars(T, lb=0.0, ub=Pmax_cha3, name="Pcha_ES3")
Pdis3 = model.addVars(T, lb=0.0, ub=Pmax_dis3, name="Pdis_ES3")
bcha3 = model.addVars(T, vtype=GRB.BINARY, name="bcha_ES3")
bdis3 = model.addVars(T, vtype=GRB.BINARY, name="bdis_ES3")

# ---- 储能：ES4 氢（kg / kg/h）----
Pcha4 = model.addVars(T, lb=0.0, ub=Pmax_cha4, name="Pcha_ES4")   # 充氢（kg/h）
Pdis4 = model.addVars(T, lb=0.0, ub=Pmax_dis4, name="Pdis_ES4")   # 放氢（kg/h）
bcha4 = model.addVars(T, vtype=GRB.BINARY, name="bcha_ES4")
bdis4 = model.addVars(T, vtype=GRB.BINARY, name="bdis_ES4")

# ---- SOC变量下界必须允许SOC[0]=0（做法A+论文S(1)=0需要）----
SOC1  = model.addVars(T, lb=0.0, ub=GRB.INFINITY, name="SOC_ES1")  # 电储能能量（MWh）
SOC2  = model.addVars(T, lb=0.0, ub=GRB.INFINITY, name="SOC_ES2")  # 热储能能量（MWh）
SOC3  = model.addVars(T, lb=0.0, ub=GRB.INFINITY, name="SOC_ES3")  # 气储能能量（MWh）
SOC4  = model.addVars(T, lb=0.0, ub=GRB.INFINITY, name="SOC_ES4")  # 氢库存（kg）

# =========================
# 4) 设备机理约束（按设备顺序：WT → PV → GT → WHB → GB → EL → HFC）
# =========================
# =========================
# 4) 设备机理约束
# =========================

# -------- 4.1 风光场景约束（二阶段）--------
for s in S:
    for t in range(T):
        model.addConstr(PWT_use[t, s] + PWT_cut[t, s] == scenarios[s]["WT"][t],name=f"WT_avail_{s}_{t}")
        model.addConstr(PPV_use[t, s] + PPV_cut[t, s] == scenarios[s]["PV"][t],name=f"PV_avail_{s}_{t}")
# -------- 4.2 一阶段设备机理约束 --------
for t in range(T):
    # GT：气 -> 电 + 热
    model.addConstr(PGT_e[t] == ηGT * PGT_g[t],name=f"GT_g2e_{t}")
    model.addConstr(PGT_h[t] == PGT_e[t] * (1.0 - ηGT - ηloss) / ηGT,name=f"GT_e2h_{t}")
    # WHB：回收GT余热
    model.addConstr(PWHB_h[t] == ηWHB * PGT_h[t],name=f"WHB_recover_{t}")
    # GB：气 -> 热
    model.addConstr(PGB_h[t] == ηGB * PGB_g[t],name=f"GB_g2h_{t}")
    # EL：电 -> 氢
    model.addConstr(HEL[t] == ηEL * PEL_e[t] / meh,name=f"EL_e2H_{t}")

    # HFC：氢 -> 电 + 热
    model.addConstr(PHFC_tot[t] == ηHFC * HHFC[t] * meh,name=f"HFC_H2P_{t}")
    model.addConstr(PHFC_e[t] == ηHFC_e * PHFC_tot[t],name=f"HFC_P2e_{t}")
    model.addConstr(PHFC_h[t] == ηHFC_h * PHFC_tot[t],name=f"HFC_P2h_{t}")
    # MR：氢 -> 天然气
    model.addConstr(PMR_g[t] == ηMR * HMR[t] * meh,name=f"MR_H2_to_gas_{t}")

# =========================
# 5) 爬坡约束（按设备顺序：GT → GB → EL → HFC；比例爬坡：±ramp * 额定值）
# =========================
for t in range(T - 1):
    # -------- GT：发电爬坡 --------
    model.addConstr(PGT_e[t+1] - PGT_e[t] <= ramp_GT * PGT_max,name=f"ramp_GT_up_{t}")  # GT上爬坡
    model.addConstr(PGT_e[t+1] - PGT_e[t] >= -ramp_GT * PGT_max,name=f"ramp_GT_dn_{t}")  # GT下爬坡
    # -------- GB：产热爬坡 --------
    model.addConstr(PGB_h[t+1] - PGB_h[t] <= ramp_GB * PGB_max,name=f"ramp_GB_up_{t}")  # GB上爬坡
    model.addConstr(PGB_h[t+1] - PGB_h[t] >= -ramp_GB * PGB_max,name=f"ramp_GB_dn_{t}")  # GB下爬坡
    # -------- EL：耗电爬坡 --------
    model.addConstr(PEL_e[t+1] - PEL_e[t] <= ramp_EL * PEL_max,name=f"ramp_EL_up_{t}")  # EL上爬坡
    model.addConstr(PEL_e[t+1] - PEL_e[t] >= -ramp_EL * PEL_max,name=f"ramp_EL_dn_{t}")  # EL下爬
    # -------- HFC：耗氢爬坡（对HHFC）--------
    model.addConstr(HHFC[t+1] - HHFC[t] <= ramp_HFC * HHFC_max, name=f"ramp_HFC_up_{t}")  # HFC上爬坡
    model.addConstr(HHFC[t+1] - HHFC[t] >= -ramp_HFC * HHFC_max,name=f"ramp_HFC_dn_{t}")  # HFC下爬坡
    # -------- MR：耗氢爬坡（对HMR）--------
    model.addConstr(HMR[t+1] - HMR[t] <= ramp_MR * HMR_max,name=f"ramp_MR_up_{t}")   # MR上爬坡
    model.addConstr(HMR[t+1] - HMR[t] >= -ramp_MR * HMR_max,name=f"ramp_MR_dn_{t}")   # MR下爬坡

from gurobipy import GRB
import math

# =========================
# 6) 储能约束（按论文贴近版）
# 说明：
# 1. 仅 ES1（电储能）考虑容量衰减，对应 Eq.(19)
# 2. ES2/ES3/ES4 不考虑容量衰减
# 3. 充放功率上限按论文 Eq.(20)：0.3 * Pmax
# 4. 首时段最低充电约束按论文 Eq.(20)：Pcha(1) >= 0.2 * Pmax
# 5. 首末库存关系按论文 Eq.(20)：S_n(1) = S_n(T) + ΔS_n
# 6. SOC 上下界：
#    - ES1：Smin1 <= SOC1[t] <= min(Smax1, Pmax1_eff[t])
#    - ES2/ES3/ES4：Smin <= SOC <= Smax
# =========================

# =========================
# 6.0 自放电率（论文：电储能0.001，其它0）
# =========================
mu_ES1 = 0.001
mu_ES2 = 0.0
mu_ES3 = 0.0
mu_ES4 = 0.0

# =========================
# 6.1 电储能容量衰减（仅 ES1，对应论文 Eq.(19)）
# Pmax1_eff[t] 作为 ES1 的可用容量上界参考
# =========================
a_deg = 200.0
b_deg = 4.168e-4
c_deg = 1.9984e-7
d_deg = 0.0332

Pmax1_eff = [0.0] * T
SOC1_ub = [0.0] * T

for t in range(T):
    tau = float(t + 1)   # 避免 sqrt(0)
    Pmax1_eff[t] = a_deg + b_deg * tau - c_deg * (tau ** 2) - d_deg * math.sqrt(tau)
    # ES1 的库存上界 = min(表2上界, 衰减后的可用容量)
    SOC1_ub[t] = min(Smax1, Pmax1_eff[t])

# =========================
# 6.3 互斥约束 + 充放电功率上限
# 按论文 Eq.(20)：0 <= Pcha <= Bcha * 0.3 * Pmax
#                0 <= Pdis <= Bdis * 0.3 * Pmax
# =========================
for t in range(T):
    # -------- ES1 电储能 --------
    model.addConstr(bcha1[t] + bdis1[t] <= 1, name=f"ES1_mutex_{t}")
    model.addConstr(Pcha1[t] <= bcha1[t] * 0.3 * Pmax_cha1, name=f"ES1_cha_cap_{t}")
    model.addConstr(Pdis1[t] <= bdis1[t] * 0.3 * Pmax_dis1, name=f"ES1_dis_cap_{t}")

    # -------- ES2 热储能 --------
    model.addConstr(bcha2[t] + bdis2[t] <= 1, name=f"ES2_mutex_{t}")
    model.addConstr(Pcha2[t] <= bcha2[t] * 0.3 * Pmax_cha2, name=f"ES2_cha_cap_{t}")
    model.addConstr(Pdis2[t] <= bdis2[t] * 0.3 * Pmax_dis2, name=f"ES2_dis_cap_{t}")

    # -------- ES3 气储能 --------
    model.addConstr(bcha3[t] + bdis3[t] <= 1, name=f"ES3_mutex_{t}")
    model.addConstr(Pcha3[t] <= bcha3[t] * 0.3 * Pmax_cha3, name=f"ES3_cha_cap_{t}")
    model.addConstr(Pdis3[t] <= bdis3[t] * 0.3 * Pmax_dis3, name=f"ES3_dis_cap_{t}")

    # -------- ES4 氢储能 --------
    model.addConstr(bcha4[t] + bdis4[t] <= 1, name=f"ES4_mutex_{t}")
    model.addConstr(Pcha4[t] <= bcha4[t] * 0.3 * Pmax_cha4, name=f"ES4_cha_cap_{t}")
    model.addConstr(Pdis4[t] <= bdis4[t] * 0.3 * Pmax_dis4, name=f"ES4_dis_cap_{t}")

# =========================
# 6.4 初始库存约束
model.addConstr(SOC1[T-1] == SOC1[0], name="ES1_terminal")
model.addConstr(SOC2[T-1] == SOC2[0], name="ES2_terminal")
model.addConstr(SOC3[T-1] == SOC3[0], name="ES3_terminal")
model.addConstr(SOC4[T-1] == SOC4[0], name="ES4_terminal")

# =========================
# 6.5 首时段最低充电约束
# 论文：Pcha_ES,n(1) >= 0.2 * Pmax_n
# =========================
model.addConstr(Pcha1[0] >= 0.2 * Pmax_cha1, name="ES1_init_charge_min")
model.addConstr(Pcha2[0] >= 0.2 * Pmax_cha2, name="ES2_init_charge_min")
model.addConstr(Pcha3[0] >= 0.2 * Pmax_cha3, name="ES3_init_charge_min")
model.addConstr(Pcha4[0] >= 0.2 * Pmax_cha4, name="ES4_init_charge_min")

# =========================
# 6.6 SOC 动态方程
# 论文：S_n(t) = (1-μ)S_n(t-1) + ηcha*Pcha - Pdis/ηdis
# 注意这里从 t=1 开始更新
# =========================
for t in range(1, T):
    # -------- ES1 电储能 --------
    model.addConstr(SOC1[t] == (1.0 - mu_ES1) * SOC1[t-1] + ηcha_ES1 * Pcha1[t]- Pdis1[t] / ηdis_ES1,name=f"ES1_SOC_dyn_{t}")
    # -------- ES2 热储能 --------
    model.addConstr(SOC2[t] == (1.0 - mu_ES2) * SOC2[t-1]+ ηcha_ES2 * Pcha2[t]- Pdis2[t] / ηdis_ES2,name=f"ES2_SOC_dyn_{t}")
    # -------- ES3 气储能 --------
    model.addConstr(SOC3[t] == (1.0 - mu_ES3) * SOC3[t-1]+ ηcha_ES3 * Pcha3[t]- Pdis3[t] / ηdis_ES3,name=f"ES3_SOC_dyn_{t}")
    # -------- ES4 氢储能 --------
    model.addConstr(SOC4[t] == (1.0 - mu_ES4) * SOC4[t-1]+ ηcha_ES4 * Pcha4[t]- Pdis4[t] / ηdis_ES4,name=f"ES4_SOC_dyn_{t}")

# =========================
# 6.7 SOC 上下界约束
# =========================
for t in range(1, T):
    # -------- ES1 电储能：考虑衰减 --------
    model.addConstr(SOC1[t] >= Smin1, name=f"ES1_SOC_min_{t}")
    model.addConstr(SOC1[t] <= SOC1_ub[t], name=f"ES1_SOC_max_{t}")

    # -------- ES2 热储能：不考虑衰减 --------
    model.addConstr(SOC2[t] >= Smin2, name=f"ES2_SOC_min_{t}")
    model.addConstr(SOC2[t] <= Smax2, name=f"ES2_SOC_max_{t}")

    # -------- ES3 气储能：不考虑衰减 --------
    model.addConstr(SOC3[t] >= Smin3, name=f"ES3_SOC_min_{t}")
    model.addConstr(SOC3[t] <= Smax3, name=f"ES3_SOC_max_{t}")

    # -------- ES4 氢储能：不考虑衰减 --------
    model.addConstr(SOC4[t] >= Smin4, name=f"ES4_SOC_min_{t}")
    model.addConstr(SOC4[t] <= Smax4, name=f"ES4_SOC_max_{t}")


# =========================
# 7) MR / CCS / 碳排放约束
# =========================
for t in range(T):

    # -------- 碳配额 --------
    model.addConstr(Egrid_q[t] == kappa_buy * Pbuy_e[t, "base"],name=f"Egrid_quota_{t}")   # 购电初始碳配额
    model.addConstr(EGT_q[t] == kappa_GT * PGT_g[t],name=f"EGT_quota_{t}")     # GT初始碳配额
    model.addConstr(EGB_q[t] == kappa_GB * PGB_g[t],name=f"EGB_quota_{t}")     # GB初始碳配额

    # -------- 实际碳排放 --------
    model.addConstr(Egrid_a[t] == kappa_buy_a * Pbuy_e[t, "base"],name=f"Egrid_actual_{t}")  # 购电实际排放
    model.addConstr(EGT_a[t] == kappa_GT_e * PGT_e[t] + kappa_GT_h * PGT_h[t],name=f"EGT_actual_{t}")    # GT实际排放
    model.addConstr(EGB_a[t] == kappa_GB_a * PGB_h[t],name=f"EGB_actual_{t}")    # GB实际排放

    # -------- GT + GB 总排放 --------
    model.addConstr(ECO2[t] == EGT_a[t] + EGB_a[t],name=f"ECO2_total_{t}")    # GT与GB产生的总CO2

    # -------- CCS 捕集 --------
    model.addConstr(Ecap_CCS[t] == ηCCS * ECO2[t],name=f"CCS_capture_{t}")   # 捕集CO2
    model.addConstr(Ecs_CCS[t] == (1 - ηCCS) * ECO2[t],name=f"CCS_uncaptured_{t}")  # 未捕集CO2
    model.addConstr(PCSSt[t] == eccs * Ecap_CCS[t],name=f"CCS_power_{t}")     # CCS耗电

    # -------- MR 消耗捕集的CO2 --------
    model.addConstr(EMR[t] == θ * HMR[t],name=f"MR_CO2_use_{t}")    # MR消耗CO2
    model.addConstr(EMR[t] <= Ecap_CCS[t],name=f"MR_CO2_limit_{t}")  # MR消耗CO2不能超过捕集量

    # -------- 封存量分解 --------
    model.addConstr(Ecap_CCS[t] == EMR[t] + ECS[t],name=f"CCS_split_{t}")     # 捕集CO2 = MR利用 + 封存

    # -------- 排入环境的CO2 --------
    model.addConstr(ECO2_env[t] == Ecs_CCS[t] + Egrid_a[t],name=f"ECO2_env_{t}")      # 环境净排放 = 未捕集排放 + 购电排放

# -------- 总初始碳配额 --------
model.addConstr(E_IES_quota == grb.quicksum(Egrid_q[t] + EGT_q[t] + EGB_q[t] for t in range(T)),
    name="Total_IES_Quota"
)

# -------- 参与碳交易的净排放量 --------
model.addConstr(E_IES_actual == grb.quicksum(ECO2_env[t] for t in range(T)) - E_IES_quota,
    name="Total_IES_Actual")

# =========================
# 8) 四能量平衡约束（仅能量鲁棒，场景相关）
# 说明：
# - 第一阶段变量：GT, GB, EL, HFC, MR, 储能等
# - 第二阶段变量：购电/购热/购气/购氢，风光利用/弃电
# =========================
for s in S:
    for t in range(T):
        # -------- 电平衡 --------
        model.addConstr(Pbuy_e[t, s] + PGT_e[t] + PHFC_e[t] + Pdis1[t] + PWT_use[t, s] + PPV_use[t, s] == electric_load[t] + PEL_e[t] + Pcha1[t] + PCSSt[t], name=f"Balance_Elec_{s}_{t}")
        # -------- 热平衡 --------
        model.addConstr(Pbuy_h[t, s] + PWHB_h[t] + PHFC_h[t] + PGB_h[t] + Pdis2[t] == thermal_load[t] + Pcha2[t], name=f"Balance_Heat_{s}_{t}")
        # -------- 气平衡 --------
        model.addConstr(Pbuy_g[t, s] + PMR_g[t] + Pdis3[t] == gas_load[t] + Pcha3[t] + PGT_g[t] + PGB_g[t], name=f"Balance_Gas_{s}_{t}")
        # -------- 氢平衡 --------
        model.addConstr(Pbuy_H[t, s] + HEL[t] + Pdis4[t] == hydrogen_load[t] + Pcha4[t] + HHFC[t] + HMR[t], name=f"Balance_H2_{s}_{t}")

# =========================
# 9) 阶梯碳交易成本约束（修改版：同时考虑“超额购买”和“盈余出售”）
# 说明：
# 1. 若净排放 E_IES_actual > 0，则需要购买碳配额，形成正的碳交易成本
# 2. 若净排放 E_IES_actual < 0，则说明存在剩余配额，可出售获利，形成负的碳交易成本
# 3. 因此，CCT 允许为负值
# 4. 这里采用“买/卖分解”的方式建模：
#       E_IES_actual = E_buy - E_sell
#    其中：
#       E_buy  >= 0   表示需要购买的配额量
#       E_sell >= 0   表示可以出售的剩余配额量
# 5. 购买部分采用阶梯碳交易价格
# 6. 出售部分为了保持线性和简洁，先按“基准碳价 λ”出售
# =========================

# -------- 买入 / 卖出碳配额变量 --------
E_buy  = model.addVar(lb=0.0, name="E_buy")    # 需要购买的碳配额量（t）
E_sell = model.addVar(lb=0.0, name="E_sell")   # 可出售的剩余碳配额量（t）
y_buy = model.addVar(vtype=GRB.BINARY, name="y_buy")
M_carbon = 1e5
model.addConstr(E_buy <= M_carbon * y_buy, name="E_buy_logic")
model.addConstr(E_sell <= M_carbon * (1 - y_buy), name="E_sell_logic")
# -------- 将净排放拆成：净排放 = 买入 - 卖出 --------
# 若 E_IES_actual > 0，则 E_buy > 0, E_sell = 0
# 若 E_IES_actual < 0，则 E_buy = 0, E_sell > 0
model.addConstr(E_IES_actual == E_buy - E_sell, name="Carbon_buy_sell_balance")

# =========================
# 9.1 购买碳配额的阶梯分段变量
# 对 E_buy 分 5 段：
#   [0, l]
#   [l, 2l]
#   [2l, 3l]
#   [3l, 4l]
#   [4l, +inf)
# 每一段价格逐级增加
# =========================
z1 = model.addVar(lb=0.0, ub=l_carbon, name="z1")   # 第1段购买量
z2 = model.addVar(lb=0.0, ub=l_carbon, name="z2")   # 第2段购买量
z3 = model.addVar(lb=0.0, ub=l_carbon, name="z3")   # 第3段购买量
z4 = model.addVar(lb=0.0, ub=l_carbon, name="z4")   # 第4段购买量
z5 = model.addVar(lb=0.0, name="z5")                # 第5段购买量（无上限）

# -------- 购买总量 = 各段之和 --------
model.addConstr(E_buy == z1 + z2 + z3 + z4 + z5, name="Carbon_buy_segment_sum")

# =========================
# 9.2 购买成本（阶梯价格）
# 对应论文中的阶梯碳交易思想：
#   第1段：λ
#   第2段：λ(1+α)
#   第3段：λ(1+2α)
#   第4段：λ(1+3α)
#   第5段：λ(1+4α)
# =========================
C_buy = model.addVar(lb=0.0, name="Carbon_buy_cost")   # 购买碳配额成本（¥）

model.addConstr(
    C_buy ==
    lambda_carbon * z1 +
    lambda_carbon * (1 + alpha_carbon) * z2 +
    lambda_carbon * (1 + 2 * alpha_carbon) * z3 +
    lambda_carbon * (1 + 3 * alpha_carbon) * z4 +
    lambda_carbon * (1 + 4 * alpha_carbon) * z5,
    name="Carbon_buy_cost_piecewise"
)

# =========================
# 9.3 出售收益
# 这里先采用“按基准碳价出售”的简化方式：
#   收益 = λ * E_sell
# 因为收益会抵减总成本，所以在总碳交易成本中表现为负项
# =========================
C_sell = model.addVar(lb=0.0, name="Carbon_sell_revenue")   # 卖出碳配额收益（¥）

model.addConstr(
    C_sell == lambda_carbon * E_sell,
    name="Carbon_sell_revenue_def"
)

# =========================
# 9.4 总碳交易成本
# 总碳交易成本 = 购买成本 - 出售收益
# 因此 CCT 可能为负（表示卖碳赚钱）
# =========================
model.addConstr(
    CCT == C_buy - C_sell,
    name="CarbonTradingCost_total"
)

# 约束部分结束

<gurobi.Constr *Awaiting Model Update*>

In [31]:
# =========================
# 10) 目标函数（能量鲁棒版）
# 说明：
# 1. 第一阶段成本：
#    - COP：设备与储能运维成本
#    - CCS_cost：碳封存成本
#    - CCT：碳交易成本（暂按 base 场景购电计算，不做场景鲁棒）
# 2. 第二阶段最坏场景成本：
#    - 外购电 / 热 / 气 / 氢成本
#    - 弃风弃光惩罚成本
# 3. 总目标：
#    min { 第一阶段固定成本 + 最坏场景二阶段成本 }
# =========================

# =========================
# 10.1 最坏场景成本变量
# eta >= 每个场景下的二阶段能量补救成本
# =========================
eta = model.addVar(lb=0.0, name="eta")

second_stage_cost = {}
for s in S:
    second_stage_cost[s] = grb.quicksum(
        electricity_price[t] * Pbuy_e[t, s] * dt +              # 外购电成本
        natural_gas_price * Pbuy_g[t, s] * dt +                 # 外购气成本
        hydrogen_price * (Pbuy_H[t, s] * meh) * dt +            # 外购氢成本（kg -> MWh）
        heat_price * Pbuy_h[t, s] * dt +                        # 外购热成本
        wind_penalty * PWT_cut[t, s] * dt +                     # 弃风惩罚
        solar_penalty * PPV_cut[t, s] * dt                      # 弃光惩罚
        for t in range(T)
    )
    model.addConstr(
        eta >= second_stage_cost[s],
        name=f"worst_cost_{s}"
    )

# =========================
# 10.2 定义成本项变量
# =========================
COP = model.addVar(lb=0.0, name="COP")                      # 第一阶段运维成本
CCS_cost = model.addVar(lb=0.0, name="CCS_cost")           # 第一阶段碳封存成本
C_total = model.addVar(lb=-GRB.INFINITY, name="C_total")   # 总成本

# （可选）仅用于结果输出展示，不参与目标函数单独建约束
CEP_base = model.addVar(lb=0.0, name="CEP_base")           # base场景外购能成本
CWS_base = model.addVar(lb=0.0, name="CWS_base")           # base场景弃风弃光成本

# =========================
# 10.3 第一阶段运维成本 COP
# 注意：
# - 这里不再包含 PWT / PPV，因为风光利用已是场景相关二阶段变量
# - 风光若要计运维，通常可视为零边际成本；若你坚持加，也建议放在各场景二阶段成本里
# =========================
model.addConstr(
    COP ==
    grb.quicksum(
        # -------- 主设备运维成本（第一阶段）--------
        αGT * PGT_e[t] * dt +
        αGB * PGB_h[t] * dt +
        αWHB * PWHB_h[t] * dt +
        αEL * PEL_e[t] * dt +
        αMR * PMR_g[t] * dt +
        αHFC * PHFC_tot[t] * dt +
        αCCS * PCSSt[t] * dt +

        # -------- 储能运维成本（第一阶段）--------
        α1 * (Pcha1[t] + Pdis1[t]) * dt +
        α2 * (Pcha2[t] + Pdis2[t]) * dt +
        α3 * (Pcha3[t] + Pdis3[t]) * dt +
        α4 * (Pcha4[t] + Pdis4[t]) * dt
        for t in range(T)
    ),
    name="Cost_COP"
)

# =========================
# 10.4 碳封存成本 CCS_cost
# =========================
model.addConstr(
    CCS_cost ==
    grb.quicksum(
        kcs * ECS[t] * dt
        for t in range(T)
    ),
    name="Cost_CCS"
)

# =========================
# 10.5 （可选）base场景成本展示项
# 这两个变量只是为了后续你输出结果更方便，不影响目标函数结构
# =========================
model.addConstr(
    CEP_base ==
    grb.quicksum(
        electricity_price[t] * Pbuy_e[t, "base"] * dt +
        natural_gas_price * Pbuy_g[t, "base"] * dt +
        hydrogen_price * (Pbuy_H[t, "base"] * meh) * dt +
        heat_price * Pbuy_h[t, "base"] * dt
        for t in range(T)
    ),
    name="Cost_CEP_base"
)

model.addConstr(
    CWS_base ==
    grb.quicksum(
        wind_penalty * PWT_cut[t, "base"] * dt +
        solar_penalty * PPV_cut[t, "base"] * dt
        for t in range(T)
    ),
    name="Cost_CWS_base"
)

# =========================
# 10.6 总成本
# 能量鲁棒版目标：
# C_total = COP + CCS_cost + CCT + eta
# 其中 eta 表示最坏场景的二阶段能量补救成本
# =========================
model.addConstr(
    C_total == COP + CCS_cost + CCT + eta,
    name="Cost_Total"
)

# =========================
# 10.7 设置目标函数
# =========================
model.setObjective(C_total, GRB.MINIMIZE)

In [32]:
import pandas as pd
from gurobipy import GRB

# =========================
# 模型求解
# =========================
model.optimize()

if model.status != GRB.OPTIMAL:
    print("未得到最优解，status =", model.status)

else:
    print("最小总成本 ObjVal =", model.ObjVal)

    def vlist(vdict, T):
        """一维变量提取为长度 T 的 list"""
        return [vdict[t].X for t in range(T)]

    def vlist_s(vdict, T, s):
        """二维场景变量提取为长度 T 的 list"""
        return [vdict[t, s].X for t in range(T)]

    # =========================
    # 1) 第一阶段结果提取
    # =========================
    res_first = pd.DataFrame({"t": list(range(T))})

    # ---- 负荷 ----
    res_first["E_Load"] = electric_load
    res_first["H_Load"] = thermal_load
    res_first["G_Load"] = gas_load
    res_first["H2_Load"] = hydrogen_load

    # ---- 主设备 ----
    res_first["GT_e"] = vlist(PGT_e, T)
    res_first["GT_g"] = vlist(PGT_g, T)
    res_first["GT_h"] = vlist(PGT_h, T)

    res_first["GB_h"] = vlist(PGB_h, T)
    res_first["GB_g"] = vlist(PGB_g, T)

    res_first["WHB_h"] = vlist(PWHB_h, T)

    res_first["EL_e"] = vlist(PEL_e, T)
    res_first["EL_H2"] = vlist(HEL, T)

    res_first["HFC_H2"] = vlist(HHFC, T)
    res_first["HFC_tot"] = vlist(PHFC_tot, T)
    res_first["HFC_e"] = vlist(PHFC_e, T)
    res_first["HFC_h"] = vlist(PHFC_h, T)

    res_first["MR_H2"] = vlist(HMR, T)
    res_first["MR_g"] = vlist(PMR_g, T)
    res_first["EMR"] = vlist(EMR, T)

    # ---- CCS / 碳 ----
    res_first["ECO2"] = vlist(ECO2, T)
    res_first["Ecap_CCS"] = vlist(Ecap_CCS, T)
    res_first["Ecs_CCS"] = vlist(Ecs_CCS, T)
    res_first["PCSSt"] = vlist(PCSSt, T)
    res_first["ECS"] = vlist(ECS, T)
    res_first["Egrid_q"] = vlist(Egrid_q, T)
    res_first["Egrid_a"] = vlist(Egrid_a, T)
    res_first["EGT_q"] = vlist(EGT_q, T)
    res_first["EGT_a"] = vlist(EGT_a, T)
    res_first["EGB_q"] = vlist(EGB_q, T)
    res_first["EGB_a"] = vlist(EGB_a, T)
    res_first["ECO2_env"] = vlist(ECO2_env, T)

    # ---- 储能 ----
    res_first["Pcha1"] = vlist(Pcha1, T)
    res_first["Pdis1"] = vlist(Pdis1, T)
    res_first["SOC1"] = vlist(SOC1, T)

    res_first["Pcha2"] = vlist(Pcha2, T)
    res_first["Pdis2"] = vlist(Pdis2, T)
    res_first["SOC2"] = vlist(SOC2, T)

    res_first["Pcha3"] = vlist(Pcha3, T)
    res_first["Pdis3"] = vlist(Pdis3, T)
    res_first["SOC3"] = vlist(SOC3, T)

    res_first["Pcha4"] = vlist(Pcha4, T)
    res_first["Pdis4"] = vlist(Pdis4, T)
    res_first["SOC4"] = vlist(SOC4, T)

    # =========================
    # 2) 各场景第二阶段结果提取
    # =========================
    scenario_results = {}

    for s in S:
        df = pd.DataFrame({"t": list(range(T))})
        df["Scenario"] = s

        df["WT_avail"] = scenarios[s]["WT"]
        df["PV_avail"] = scenarios[s]["PV"]

        df["PWT_use"] = vlist_s(PWT_use, T, s)
        df["PWT_cut"] = vlist_s(PWT_cut, T, s)

        df["PPV_use"] = vlist_s(PPV_use, T, s)
        df["PPV_cut"] = vlist_s(PPV_cut, T, s)

        df["Pbuy_e"] = vlist_s(Pbuy_e, T, s)
        df["Pbuy_g"] = vlist_s(Pbuy_g, T, s)
        df["Pbuy_h"] = vlist_s(Pbuy_h, T, s)
        df["Pbuy_H"] = vlist_s(Pbuy_H, T, s)

        scenario_results[s] = df

    # =========================
    # 3) 各场景成本统计
    # =========================
    scenario_cost_table = []

    for s in S:
        cost_buy_e = sum(electricity_price[t] * Pbuy_e[t, s].X * dt for t in range(T))
        cost_buy_g = sum(natural_gas_price * Pbuy_g[t, s].X * dt for t in range(T))
        cost_buy_h = sum(heat_price * Pbuy_h[t, s].X * dt for t in range(T))
        cost_buy_H = sum(hydrogen_price * (Pbuy_H[t, s].X * meh) * dt for t in range(T))

        cost_curt_wt = sum(wind_penalty * PWT_cut[t, s].X * dt for t in range(T))
        cost_curt_pv = sum(solar_penalty * PPV_cut[t, s].X * dt for t in range(T))

        cost_energy = cost_buy_e + cost_buy_g + cost_buy_h + cost_buy_H
        cost_curt = cost_curt_wt + cost_curt_pv
        total_second_stage = cost_energy + cost_curt

        scenario_cost_table.append({
            "Scenario": s,
            "BuyElectricity": cost_buy_e,
            "BuyGas": cost_buy_g,
            "BuyHeat": cost_buy_h,
            "BuyHydrogen": cost_buy_H,
            "CurtailWind": cost_curt_wt,
            "CurtailPV": cost_curt_pv,
            "SecondStageTotal": total_second_stage
        })

    scenario_cost_df = pd.DataFrame(scenario_cost_table)

    # =========================
    # 4) 第一阶段成本统计
    # =========================
    cost_device_om = sum(
        αGT * PGT_e[t].X * dt +
        αGB * PGB_h[t].X * dt +
        αWHB * PWHB_h[t].X * dt +
        αEL * PEL_e[t].X * dt +
        αMR * PMR_g[t].X * dt +
        αHFC * PHFC_tot[t].X * dt +
        αCCS * PCSSt[t].X * dt
        for t in range(T)
    )

    cost_storage_om = sum(
        α1 * (Pcha1[t].X + Pdis1[t].X) * dt +
        α2 * (Pcha2[t].X + Pdis2[t].X) * dt +
        α3 * (Pcha3[t].X + Pdis3[t].X) * dt +
        α4 * (Pcha4[t].X + Pdis4[t].X) * dt
        for t in range(T)
    )

    cost_COP_manual = cost_device_om + cost_storage_om
    cost_CCS_manual = sum(kcs * ECS[t].X * dt for t in range(T))

    # =========================
    # 5) 汇总输出
    # =========================
    print("\n========== 鲁棒优化成本汇总 ==========")
    print(f"调度总成本 / 元: {model.ObjVal:.4f}")
    print(f"第一阶段运维成本 COP / 元: {COP.X:.4f}")
    print(f"手工核算设备运维成本 / 元: {cost_device_om:.4f}")
    print(f"手工核算储能运维成本 / 元: {cost_storage_om:.4f}")
    print(f"手工核算总运维成本 / 元: {cost_COP_manual:.4f}")
    print(f"碳封存成本 CCS_cost / 元: {CCS_cost.X:.4f}")
    print(f"手工核算碳封存成本 / 元: {cost_CCS_manual:.4f}")
    print(f"碳交易成本 CCT / 元: {CCT.X:.4f}")
    print(f"最坏场景二阶段成本 eta / 元: {eta.X:.4f}")
    print(f"Base场景购能成本 CEP_base / 元: {CEP_base.X:.4f}")
    print(f"Base场景弃风弃光成本 CWS_base / 元: {CWS_base.X:.4f}")
    print(f"总成本 C_total / 元: {C_total.X:.4f}")

    print("\n========== 各场景二阶段成本 ==========")
    print(scenario_cost_df.to_string(index=False))

    # 找到最坏场景（按手工统计）
    worst_idx = scenario_cost_df["SecondStageTotal"].idxmax()
    worst_scenario = scenario_cost_df.loc[worst_idx, "Scenario"]
    worst_value = scenario_cost_df.loc[worst_idx, "SecondStageTotal"]

    print("\n========== 最坏场景识别 ==========")
    print(f"最坏场景: {worst_scenario}")
    print(f"最坏场景二阶段成本 / 元: {worst_value:.4f}")



最小总成本 ObjVal = 798860.800182706

========== 鲁棒优化成本汇总 ==========
调度总成本 / 元: 798860.8002
第一阶段运维成本 COP / 元: 134009.1556
手工核算设备运维成本 / 元: 125769.0432
手工核算储能运维成本 / 元: 8240.1124
手工核算总运维成本 / 元: 134009.1556
碳封存成本 CCS_cost / 元: 266.0301
手工核算碳封存成本 / 元: 266.0301
碳交易成本 CCT / 元: -127721.2402
最坏场景二阶段成本 eta / 元: 792306.8546
Base场景购能成本 CEP_base / 元: 783730.5841
Base场景弃风弃光成本 CWS_base / 元: 8576.2705
总成本 C_total / 元: 798860.8002

========== 各场景二阶段成本 ==========
   Scenario  BuyElectricity        BuyGas  BuyHeat  BuyHydrogen  CurtailWind  CurtailPV  SecondStageTotal
       base        0.000000 781052.184119      0.0       2678.4  4936.270528     3640.0     792306.854647
  wt_robust     4434.452347 781052.184119      0.0       2678.4   501.818182     3640.0     792306.854647
  pv_robust     4141.818182 781052.184119      0.0       2678.4  4434.452347        0.0     792306.854647
both_robust     8576.270528 781052.184119      0.0       2678.4     0.000000        0.0     792306.854647

========== 最坏场景识别 ======